In [2]:
from huggingface_hub import login
from dotenv import load_dotenv
from os import getenv

# login huggingface hub
load_dotenv("../.envs")
login(token=getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:

from huggingface_hub import whoami

# check our organization
for org in whoami()["orgs"]:
    for k, v in org.items():
        print(k, v)
    print("-" * 20)
    # 아래에서 name이 organization name을 의미 : natural-beauty

type org
id 673584f79621af686bbb755a
name UpstageShareKbank
fullname UpstageShareKbank
email None
canPay False
periodEnd None
avatarUrl https://www.gravatar.com/avatar/83cd1d092a6ffd81484047dbfd280201?d=retro&size=100
roleInOrg read
isEnterprise False
--------------------
type org
id 681b0cb0dba891d54be0773d
name mcp-course
fullname Hugging Face MCP Course
email None
canPay False
periodEnd None
avatarUrl https://cdn-avatars.huggingface.co/v1/production/uploads/62d648291fa3e4e7ae3fa6e8/itgTDqMrnvgNfJZJ4YmCt.png
roleInOrg read
isEnterprise False
--------------------
type org
id 687c7d1ea565d4670d466e64
name natural-beauty
fullname ADAC4T10
email None
canPay False
periodEnd None
avatarUrl https://cdn-avatars.huggingface.co/v1/production/uploads/63aa56663453852ef543a7e8/dU3v3xexLgfFDh9UoxvIQ.png
roleInOrg admin
isEnterprise False
--------------------


In [4]:
REPO_ID = "687c7d1ea565d4670d466e64"

In [5]:
from typing import Literal, Any
from pathlib import Path

## load example data
dataset_path = "example_data"
dataset_dir = Path(dataset_path).resolve()

splits = ["train", "val", "test"]
file_sufficx = ".tsv"

dataset: dict[
    Literal["train", "val", "test"],
    list[
        dict[str, Any]
    ]
] = dict()

LABELS = ['불평/불만',
 '환영/호의',
 '감동/감탄',
 '지긋지긋',
 '고마움',
 '슬픔',
 '화남/분노',
 '존경',
 '기대감',
 '우쭐댐/무시함',
 '안타까움/실망',
 '비장함',
 '의심/불신',
 '뿌듯함',
 '편안/쾌적',
 '신기함/관심',
 '아껴주는',
 '부끄러움',
 '공포/무서움',
 '절망',
 '한심함',
 '역겨움/징그러움',
 '짜증',
 '어이없음',
 '없음',
 '패배/자기혐오',
 '귀찮음',
 '힘듦/지침',
 '즐거움/신남',
 '깨달음',
 '죄책감',
 '증오/혐오',
 '흐뭇함(귀여움/예쁨)',
 '당황/난처',
 '경악',
 '부담/안_내킴',
 '서러움',
 '재미없음',
 '불쌍함/연민',
 '놀람',
 '행복',
 '불안/걱정',
 '기쁨',
 '안심/신뢰']


for split in splits:
    dataset[split] = []
    with open(dataset_dir / f"{split}{file_sufficx}", "r") as f:
        lines = f.readlines()
    
    for line in lines:
        text_id, text, multi_label = line.split("\t")
        original_labels = [int(label) for label in multi_label.strip().split(",")]
        original_label_texts = [LABELS[label] for label in original_labels]
        dataset[split].append({
            "id": text_id, "text": text, "original_labels": original_labels, "original_label_texts": original_label_texts
        })

dataset["train"][0]

{'id': '39087',
 'text': '내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.',
 'original_labels': [2, 13, 15, 16, 29, 39],
 'original_label_texts': ['감동/감탄', '뿌듯함', '신기함/관심', '아껴주는', '깨달음', '놀람']}

In [6]:
## convert multi-label to single-label

# spicy level
SENTIMENT_LABELS = ["긍정", "중립", "부정", "매우 부정"]
SPICY_LEVEL = {
    0 : [
        "환영/호의",
        "감동/감탄",
        "고마움",
        "존경",
        "기대감",
        "뿌듯함",
        "편안/쾌적",
        "신기함/관심",
        "아껴주는",
        "즐거움/신남",
        "흐뭇함(귀여움/예쁨)",
        "행복",
        "기쁨",
        "안심/신뢰",
    ], # 긍정
    1 : [
        "놀람",
        "깨달음",
        "비장함",
        "없음",
    ], # 중립
    2 : [
        "슬픔",
        "안타까움/실망",
        "부끄러움",
        "힘듦/지침",
        "죄책감",
        "당황/난처",
        "부담/안_내킴",
        "서러움",
        "재미없음",
        "불쌍함/연민",
        "불안/걱정",
    ], # 부정
    3 : [
        "불평/불만",
        "지긋지긋",
        "화남/분노",
        "의심/불신",
        "공포/무서움",
        "절망",
        "한심함",
        "역겨움/징그러움",
        "짜증",
        "어이없음",
        "패배/자기혐오",
        "귀찮음",
        "증오/혐오",
        "경악",
        "우쭐댐/무시함",
    ] # 매우 부정
}
print("긍정 : ", len(SPICY_LEVEL[0]))
print("중립 : ", len(SPICY_LEVEL[1]))
print("부정 : ", len(SPICY_LEVEL[2]))
print("매우 부정 : ", len(SPICY_LEVEL[3]))
print("total : ", len(SPICY_LEVEL[0]) + len(SPICY_LEVEL[1]) + len(SPICY_LEVEL[2]) + len(SPICY_LEVEL[3]))

def select_spicy_level(original_labels: list[int]):
    assert len(original_labels) > 0, ValueError("original_labels must be greater than 0")
    
    label_spicy_levels = []
    for label in original_labels:
        label_text = LABELS[label]
        
        label_spicy_level = len(SPICY_LEVEL)
        for spicy_level, original_spicy_labels in SPICY_LEVEL.items():
            if label_text in original_spicy_labels:
                label_spicy_level = spicy_level
                break
        
        assert label_spicy_level != len(SPICY_LEVEL), ValueError("label spicy level not found")
        
        label_spicy_levels.append(label_spicy_level)
    
    ## 기준 : 빈도수 기반 spicy level 선택
    label_spicy_levels_count = {spicy_level: 0 for spicy_level in range(len(SPICY_LEVEL))}
    for spicy_level in label_spicy_levels:
        label_spicy_levels_count[spicy_level] += 1
    
    label_spicy_levels_count_filtered = {level: count for level, count in label_spicy_levels_count.items() if count > 0}
    
    if len(label_spicy_levels_count_filtered) == 0:
        raise ValueError("label spicy level not found")
    
    label_spicy_levels_count_sorted = sorted(label_spicy_levels_count_filtered.items(), key=lambda x: x[1], reverse=True)
    
    if len(label_spicy_levels_count_sorted) == 1 or label_spicy_levels_count_sorted[0][0] != label_spicy_levels_count_sorted[1][0]:
        spicy_label = label_spicy_levels_count_sorted[0][0]
        # spicy level에 따른 label texts
        filtered_original_label_texts = []
        for idx, original_label in enumerate(original_labels):
            if label_spicy_levels[idx] == spicy_label:
                filtered_original_label_texts.append(LABELS[original_label])
        
        return spicy_label, filtered_original_label_texts
    else:
        # 만약, 가장 높은 spicy level과 두 번째로 높은 spicy level이 같다면, 해당 문장이 감정적으로 명확하지 않다는 것을 의미
        # 이러한 데이터는 제외
        return None, None

def convert_multi_label_to_single_label(dataset, splits=["train", "val", "test"]):
    from copy import deepcopy
    
    new_dataset = deepcopy(dataset)
    for split in splits:
        for row in new_dataset[split]:
            original_labels = row["original_labels"]
            spicy_level, filtered_original_label_texts = select_spicy_level(original_labels)
            
            if spicy_level is None:
                continue

            row["label"] = spicy_level
            row["filtered_original_label_texts"] = filtered_original_label_texts
    
    return new_dataset

spicy_dataset = convert_multi_label_to_single_label(dataset)
print(spicy_dataset["train"][0])
for split in ["train", "val", "test"]:
    print(split)
    print("Before : ", len(dataset[split]))
    print("After : ", len(spicy_dataset[split]))

긍정 :  14
중립 :  4
부정 :  11
매우 부정 :  15
total :  44
{'id': '39087', 'text': '내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.', 'original_labels': [2, 13, 15, 16, 29, 39], 'original_label_texts': ['감동/감탄', '뿌듯함', '신기함/관심', '아껴주는', '깨달음', '놀람'], 'label': 0, 'filtered_original_label_texts': ['감동/감탄', '뿌듯함', '신기함/관심', '아껴주는']}
train
Before :  40000
After :  40000
val
Before :  5000
After :  5000
test
Before :  5000
After :  5000


In [7]:
from datasets import Dataset, DatasetDict, ClassLabel, Features, Value, Sequence

# 2. 데이터를 허깅페이스 허브 형식으로 변환합니다.
"""
Before :
[
    {
        "text": "",
        "label": 0
    },
    {
        "text": "",
        "label": 0
    },
    ...
]

After :
{
    "text": ["", "", ...],
    "label": [0, 0, ...]
}

"""

spicy_dataset_for_hf_train = {"id": [], "text": [], "label": [], "filtered_original_label_text": [], "original_label_text": [], "original_label": []}
spicy_dataset_for_hf_val = {"id": [], "text": [], "label": [], "filtered_original_label_text": [], "original_label_text": [], "original_label": []}
spicy_dataset_for_hf_test = {"id": [], "text": [], "label": [], "filtered_original_label_text": [], "original_label_text": [], "original_label": []}
for row in spicy_dataset["train"]:
    spicy_dataset_for_hf_train["id"].append(row["id"])
    spicy_dataset_for_hf_train["text"].append(row["text"])
    spicy_dataset_for_hf_train["label"].append(row["label"])
    spicy_dataset_for_hf_train["filtered_original_label_text"].append(row["filtered_original_label_texts"])
    spicy_dataset_for_hf_train["original_label_text"].append(row["original_label_texts"])
    spicy_dataset_for_hf_train["original_label"].append(row["original_labels"])
for row in spicy_dataset["val"]:
    spicy_dataset_for_hf_val["id"].append(row["id"])
    spicy_dataset_for_hf_val["text"].append(row["text"])
    spicy_dataset_for_hf_val["label"].append(row["label"])
    spicy_dataset_for_hf_val["filtered_original_label_text"].append(row["filtered_original_label_texts"])
    spicy_dataset_for_hf_val["original_label_text"].append(row["original_label_texts"])
    spicy_dataset_for_hf_val["original_label"].append(row["original_labels"])
for row in spicy_dataset["test"]:
    spicy_dataset_for_hf_test["id"].append(row["id"])
    spicy_dataset_for_hf_test["text"].append(row["text"])
    spicy_dataset_for_hf_test["label"].append(row["label"])
    spicy_dataset_for_hf_test["filtered_original_label_text"].append(row["filtered_original_label_texts"])
    spicy_dataset_for_hf_test["original_label_text"].append(row["original_label_texts"])
    spicy_dataset_for_hf_test["original_label"].append(row["original_labels"])



features = Features({
    "id": Value("int32"),
    "text": Value("string"),
    "label": ClassLabel(names=SENTIMENT_LABELS),
    "filtered_original_label_text": Sequence(Value("string")),
    "original_label_text": Sequence(Value("string")),
    "original_label": Sequence(Value("int32")),
})

train_dataset = Dataset.from_dict(spicy_dataset_for_hf_train, features=features)
val_dataset = Dataset.from_dict(spicy_dataset_for_hf_val, features=features)
test_dataset = Dataset.from_dict(spicy_dataset_for_hf_test, features=features)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'filtered_original_label_text', 'original_label_text', 'original_label'],
        num_rows: 40000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'filtered_original_label_text', 'original_label_text', 'original_label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'filtered_original_label_text', 'original_label_text', 'original_label'],
        num_rows: 5000
    })
})

In [8]:
organization = "natural-beauty"
dataset_name = "spicy-4class-sequence-classification-dataset"

repo_name = f"{organization}/{dataset_name}" if organization else dataset_name

# Push to hub
repo_url = dataset_dict.push_to_hub(
    repo_name,
    private=True,
    token=getenv("HF_TOKEN"),
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/40 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/665k [00:00<?, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/657k [00:00<?, ?B/s]

# Spicy 4-Class Sequence Classification Dataset

## Dataset Description

This dataset contains text samples labeled with emotion categories that have been converted into a 4-class "spicy level" classification system. The dataset is designed for sentiment analysis and emotion classification tasks.

### Dataset Structure

The dataset consists of 50,000 text samples split into:
- **Train**: 40,000 samples
- **Validation**: 5,000 samples
- **Test**: 5,000 samples

Each sample contains:
- `id`: Unique identifier for the text
- `text`: The text content
- `label`: The spicy level classification (0-3)
- `filtered_original_label_text`: Original emotion labels that match the assigned spicy level
- `original_label_text`: All original emotion labels
- `original_label`: Numeric IDs of the original emotion labels

### Spicy Level Classification

The dataset uses a 4-class classification system based on emotion intensity:

| Spicy Level | Description | Count of Emotion Categories |
|-------------|-------------|----------------------------|
| 0 | Positive | 14 emotion categories |
| 1 | Neutral | 4 emotion categories |
| 2 | Negative | 11 emotion categories |
| 3 | Very Negative | 15 emotion categories |

### Original Emotion Categories

The dataset was originally annotated with 44 fine-grained emotion categories:

#### Positive (Level 0)
- 환영/호의 (Welcome/Goodwill)
- 감동/감탄 (Moved/Admiration)
- 고마움 (Gratitude)
- 존경 (Respect)
- 기대감 (Anticipation)
- 뿌듯함 (Satisfaction)
- 편안/쾌적 (Comfort/Pleasant)
- 신기함/관심 (Curiosity/Interest)
- 아껴주는 (Cherishing)
- 즐거움/신남 (Joy/Excitement)
- 흐뭇함(귀여움/예쁨) (Fondness/Cuteness/Beauty)
- 행복 (Happiness)
- 기쁨 (Joy)
- 안심/신뢰 (Relief/Trust)

#### Neutral (Level 1)
- 놀람 (Surprise)
- 깨달음 (Realization)
- 비장함 (Solemnity)
- 없음 (None)

#### Negative (Level 2)
- 슬픔 (Sadness)
- 안타까움/실망 (Regret/Disappointment)
- 부끄러움 (Shame)
- 힘듦/지침 (Difficulty/Fatigue)
- 죄책감 (Guilt)
- 당황/난처 (Embarrassment/Awkwardness)
- 부담/안_내킴 (Burden/Reluctance)
- 서러움 (Sorrow)
- 재미없음 (Boredom)
- 불쌍함/연민 (Pity/Compassion)
- 불안/걱정 (Anxiety/Worry)

#### Very Negative (Level 3)
- 불평/불만 (Complaint/Dissatisfaction)
- 지긋지긋 (Fed up)
- 화남/분노 (Anger/Rage)
- 의심/불신 (Doubt/Distrust)
- 공포/무서움 (Fear/Terror)
- 절망 (Despair)
- 한심함 (Contempt)
- 역겨움/징그러움 (Disgust/Revulsion)
- 짜증 (Irritation)
- 어이없음 (Absurdity)
- 패배/자기혐오 (Defeat/Self-hatred)
- 귀찮음 (Annoyance)
- 증오/혐오 (Hatred/Aversion)
- 경악 (Shock)
- 우쭐댐/무시함 (Arrogance/Ignoring)

## Usage

This dataset can be used for various NLP tasks:
- Sentiment analysis
- Emotion classification
- Text classification

### Loading the Dataset

```python
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("natural-beauty/spicy-4class-sequence-classification-dataset")

# Access splits
train_data = dataset["train"]
validation_data = dataset["validation"]
test_data = dataset["test"]

# Example usage
for i, example in enumerate(train_data):
    if i >= 3:  # Show first 3 examples
        break
    print(f"Text: {example['text']}")
    print(f"Label: {example['label']}")
    print(f"Filtered emotions: {example['filtered_original_label_text']}")
    print("---")